# 0.44 — Schema 0.4: estrazione e leakage temporale

Questo notebook definisce lo **schema 0.4** (solo nodi e archi, senza eventi) e lo mette alla prova con due esperimenti:

- **E1 — qualità.** Su 50 news Bloomberg reali del 2024, quanto bene GPT-4o, GPT-5.5, il modello open Qwen3-30B-A3B e
  GLiNER2.5 estraggono nodi e archi?
- **E2 — leakage.** Su 50 news inventate, datate prima di un fatto noto oggi, i modelli scrivono cose dal futuro?
  E la data "corrente" nel prompt aiuta?

**Indice**

1. [Sintesi dei risultati](#sintesi)
2. [Lo schema 0.4: nodi, archi, attributi](#schema)
3. [I prompt](#prompt)
4. [Prova tu](#prova)
5. [E1 — qualità dell'estrazione](#e1): [risultati](#e1-risultati) · [per tipo](#e1-tipo) · [esempi di errori](#e1-errori) · [confronto con la 0.3](#e1-confronto)
6. [E2 — leakage](#e2): [risultati](#e2-risultati) · [esempi di leak](#e2-esempi)
7. [Conclusioni](#conclusioni) · [tempi della distillation](#distillation)
8. [Da decidere](#decidere)
9. [Come è stato fatto](#metodo)
10. [Dati completi](#dati)

"0.3" e "0.4" sono **versioni dello schema** (la 0.3 è definita in 0.43, la 0.4 qui), non numeri di notebook.
Il notebook legge solo risultati già salvati: gira in pochi secondi, senza chiamate API e senza modelli locali accesi.
In VS Code l'indice è anche nella vista *Outline*.

<a id="sintesi"></a>
## 1. Sintesi dei risultati

**E1 — 50 news reali, schema 0.4.** F1 = media armonica di precision e recall (100% = perfetto).

| Sistema | Nodi F1 | Archi F1 | Archi prodotti ma non nel gold | Errori totali, schema 0.3 → 0.4 |
|---|---|---|---|---|
| GPT-4o | 94% | 72% | 17 | 74 → 31 |
| GPT-5.5 | **96%** | **91%** | 5 | 61 → **12** |
| Qwen3-30B-A3B (open, su questo Mac) | 78% | 49% | 54 | 227 → 102 |
| GLiNER2.5 inglese | 63% | 26% | 28 | 131 → 129 |
| GLiNER2.5 multilingue | 54% | 17% | 61 | 171 → 187 |

**E2 — 50 news trappola.** Quante volte il modello scrive un fatto successivo alla data della news.

| Cosa chiediamo | GPT-4o | GPT-5.5 | Qwen |
|---|---|---|---|
| "Cosa è successo dopo?" — conosce il fatto futuro? | 38 | 42 | 34 |
| Estrazione con nome ufficiale e ticker — leak senza / con data | 6 / 1 | 8 / 1 | 6 / 3 |
| **Il nostro estrattore (schema 0.4)** — leak senza / con data | **0 / 0** | **0 / 0** | **0 / 0** |

- **Con lo schema 0.4 gli errori calano molto per i modelli linguistici** (GPT-5.5 da 61 a 12, GPT-4o da 74 a 31, Qwen da
  227 a 102): spariscono quelli su eventi e indicatori. I collegamenti fra aziende e prodotti/concetti restano: la recall
  degli archi è 92–100%.
- **GPT-5.5 diventa nettamente il migliore**: archi 91% contro 72% di GPT-4o, differenza significativa. GPT-4o usa male la
  freccia nuova `EXPOSED_TO` (8 frecce false) e rovescia un'acquisizione (Oaktree venditore scritto come compratore).
- **Qwen e GLiNER restano indietro**: Qwen mette `EXPOSED_TO` quasi ovunque (23 frecce false); GLiNER non migliora.
- **Nessun leak dal nostro estrattore** in 300 risposte, con o senza data. Il futuro entra solo se chiediamo nome ufficiale
  e ticker.
- **Limite**: le regole nuove della 0.4 sono state decise guardando gli output su queste stesse news, e nessuna delle 50
  contiene un `EXPOSED_TO` vero. Serve un test su news nuove.

<a id="schema"></a>
## 2. Lo schema 0.4: nodi, archi, attributi

Il grafo contiene solo **nodi** (le cose), **archi** (frecce fra due nodi, ciascuna con un tipo) e **attributi** scritti su
ogni arco. Il **tema non è un nodo**: è ciò che vogliamo scoprire, cioè un gruppo di prodotti e concetti a cui, nel tempo,
si collegano sempre più aziende, e di settori diversi. L'estrattore non scrive mai il nome di un tema che non è nella news.

**Nodi**

| Tipo | Cosa è | Esempio | Ruolo nella scoperta dei temi |
|---|---|---|---|
| COMPANY | azienda, banca, fondo | Nvidia, SMBC | le aziende di cui si misura l'adesione a un gruppo |
| PRODUCT | bene o servizio, commodity inclusa | AI chips, camion, SAF | mattoni: i gruppi di questi nodi sono i temi candidati |
| CONCEPT | tecnologia o idea | intelligenza artificiale, cattura della CO2 | mattoni, come PRODUCT |
| SECTOR | industria o mercato | difesa, real estate | misura quanti settori diversi un gruppo attraversa |
| ORGANIZATION | governo, regolatore, banca centrale | FAA, esercito italiano | solo contesto |
| LOCATION | paese, città, regione | Egitto, USA | solo contesto |

Non sono nodi: persone, titoli finanziari, importi, variabili finanziarie (ricavi, ordini, quote, prezzi), documenti,
gruppi generici, e le nazionalità ("Italy's", "German").

**Archi**

| Tipo | Da → a | Quando si usa | Esempio dalle 50 news |
|---|---|---|---|
| `OFFERS` | azienda → prodotto/concetto | lancia, produce, vende, vince un ordine per fornirlo | SMBC ─▶ Private Credit Fund |
| `ACTIVE_IN` | azienda → prodotto/concetto/settore/luogo | un descrittore o lo scopo dichiarato di un accordo | Remy ─▶ Cognac ("Cognac Maker"); Havells ─▶ Kitchen Appliances |
| `EXPOSED_TO` | azienda o settore → prodotto/concetto | la news dice che si muove *per* quel prodotto o concetto ("on", "as") | Nvidia ─▶ AI chips ("Soars on AI Chip Demand") |
| `INVESTS_IN` | azienda → prodotto/concetto/settore **oppure** → azienda | investe, o compra una quota di minoranza | Grupo Bahia ─▶ SAF; SM Trust AM ─▶ United Arrows |
| `ACQUIRES` | azienda → azienda | compra o prende il controllo | Centerbridge ─▶ Banca Progetto |
| `PARTNERS_WITH` | azienda ↔ azienda | accordo, alleanza, JV (senza direzione) | Telkom Indonesia ↔ Huawei |
| `SUPPLIES_TO` | azienda → azienda/organizzazione | fornitore → cliente con nome | Iveco ─▶ Italy Army |

I primi quattro collegano le aziende ai mattoni dei temi: sono le frecce che fanno crescere un gruppo. Gli ultimi
quattro (con `INVESTS_IN` verso un'azienda) mostrano come un gruppo si allarga: chi compra, finanzia o rifornisce chi.

**Attributi** (su ogni arco)

| Attributo | Valori | A cosa serve |
|---|---|---|
| data | la data della news | rende il grafo temporale: si contano le frecce verso un gruppo mese per mese |
| modalità | fatto · piano · incerto · negato | un piano, un'opinione ("secondo Pippo") o una smentita non pesano come un fatto |
| evidenza | la frase copiata dalla news | ogni freccia si può verificare |
| news | id del titolo | si contano le fonti indipendenti e si eliminano i doppioni |

**Rispetto alla 0.3** ([0.43](0.43-minimal-thematic-knowledge-graph-schema.ipynb), che resta invariato) spariscono il nodo
EVENT con le frecce ACTOR/TARGET, il nodo INDICATOR, gli 8 driver, trigger e descrizione; `IMPACTS` fra tipi qualsiasi
diventa `EXPOSED_TO`, solo da aziende o settori verso prodotti e concetti. Una news senza relazioni (per esempio "Apple
sale del 5%, secondo Pippo") produce solo i suoi nodi.

In [1]:
# Calcolo: legge le risposte salvate e prepara tutti i risultati usati sotto.
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(Path("../scripts").resolve()))
import extract_044 as X
import eval_044 as S

X.RUN_API = False                                   # solo risultati salvati
pd.set_option("display.max_colwidth", 200)
real = X.load_jsonl("real_50.jsonl")
gold = {g["id"]: g for g in X.load_jsonl("real_50_gold_v04.jsonl")}
probes = X.load_jsonl("synthetic_50_gold.jsonl")

def load_gliner(f):
    rows = [json.loads(l) for l in open("../data/processed/" + f)]
    return {(r["set"], r["id"]): r for r in rows[1:]}

GLINER = {"GLiNER inglese": load_gliner("gliner_044_v04.jsonl"), "GLiNER multilingue": load_gliner("gliner_044_v04_multi.jsonl")}
SYSTEMS = X.MODELS + list(GLINER)

# E1 · schema 0.4: estrazioni, conteggi per tipo di nodo/arco, lista degli errori
outs = {m: {r["id"]: X.extract_v04(r["headline"], r["date"], m, True) for r in real} for m in X.MODELS}
outs |= {g: {r["id"]: GLINER[g][("real", r["id"])] for r in real} for g in GLINER}
BD = {s: [S.breakdown_v04(outs[s][r["id"]], gold[r["id"]]) for r in real] for s in SYSTEMS}
ERR = pd.DataFrame([dict(sistema=s, id=r["id"], headline=r["headline"], **e)
                    for s in SYSTEMS for r, (_, errs, _) in zip(real, BD[s]) for e in errs])

# E2: per ogni richiesta e modello, in quali trappole compare il futuro
E2 = {}
for m in X.MODELS:
    E2[("diretta", m, None)] = [bool(S.knows_future(X.knowledge(p["headline"], p["as_of"], m), p)) for p in probes]
    for d in (False, True):
        E2[("ticker", m, d)] = [S.leak_flags_naive(X.extract_naive(p["headline"], p["as_of"], m, d), p)["any"] for p in probes]
        E2[("estrattore", m, d)] = [S.leak_flags(S.as_v03(X.extract_v04(p["headline"], p["as_of"], m, d)), p)["any"] for p in probes]
for g in GLINER:
    E2[("estrattore", g, None)] = [S.leak_flags(S.as_v03(GLINER[g][("synthetic", p["id"])]), p)["any"] for p in probes]

pct = lambda x: "–" if x != x else f"{x:.0%}"
print(f"{len(real)} news reali, {len(probes)} trappole, sistemi: {', '.join(SYSTEMS)}")

50 news reali, 50 trappole, sistemi: gpt-4o, gpt-5.5, qwen3-30b-a3b, GLiNER inglese, GLiNER multilingue


<a id="prompt"></a>
## 3. I prompt

L'estrattore (GPT e Qwen) riceve: tre righe di istruzione, la guideline 044-v2 dello schema 0.4 (la stessa usata per
scrivere il gold) e quattro esempi inventati. Il messaggio utente contiene solo la news e, se richiesto, la data.

In [2]:
msgs = X.messages_v04("Visa to Buy Fintech Startup Plaid for $5.3 Billion", "2020-01-13", dated=True)
print("── SYSTEM ──\n" + msgs[0]["content"])
print("\n── USER, con data ──\n" + msgs[1]["content"])
print("\n── USER, senza data ──\n" + X.messages_v04("Visa to Buy Fintech Startup Plaid for $5.3 Billion", dated=False)[1]["content"])

── SYSTEM ──
Extract the named nodes and the edges explicitly stated in the headline.
Copy mentions verbatim; quote evidence.
A headline with no relation gives only its nodes.

GUIDELINE 044-v2 — schema 0.4 (defined in notebook 0.44). The gold annotator and the models read this same text.

TASK. For one news headline and its date, list the named things (nodes) and the relations between them (edges) that the headline itself states. Use only the headline and what was known on its date. Never add tickers, modern names, parent companies, counterparties, sectors, products or outcomes from memory. A headline with no relation gives only its nodes; empty lists are valid.

1 · NODES — every named thing of these 6 types. `mention` is copied verbatim from the headline (drop a possessive 's).
- COMPANY: firms, banks, brokers, asset managers, funds as investing organisations, news agencies.
- ORGANIZATION: governments, ministries, regulators, central banks, agencies, courts, armed forces, universit

<a id="prova"></a>
## 4. Prova tu

Cambia news, data e modello. Una news nuova fa **una** chiamata (poi resta salvata). Qwen risponde solo se
`llama-server` è acceso (comando in [Come è stato fatto](#metodo)).

In [3]:
HEADLINE = "Nvidia Soars as Gaming Chip Sales Beat Estimates"
AS_OF = "2016-05-13"          # None = senza data
MODEL = "gpt-4o"              # "gpt-4o", "gpt-5.5" o "qwen3-30b-a3b"

X.RUN_API = True
out = X.extract_v04(HEADLINE, AS_OF, MODEL, dated=AS_OF is not None)
X.RUN_API = False
names = {n["id"]: n["mention"] for n in out["nodes"]}
print("NODI :", ", ".join(f"{n['mention']} [{n['type']}]" for n in out["nodes"]))
for e in out["edges"]:
    print("ARCO :", names.get(e["subject_id"]), f"—{e['predicate']}→", names.get(e["object_id"]), f"[{e['modality']}]", f"«{e['evidence']}»")
if not out["edges"]:
    print("ARCHI: nessuno")

NODI : Nvidia [COMPANY], Gaming Chip [PRODUCT]
ARCO : Nvidia —EXPOSED_TO→ Gaming Chip [reported] «Nvidia Soars as Gaming Chip Sales Beat Estimates»


<a id="e1"></a>
## 5. E1 — Qualità dell'estrazione su 50 news reali

50 news Bloomberg del 2024: 25 prese a caso e 25 scelte con parole-chiave legate ai predicati. Per ognuna c'è un gold
con i nodi e gli archi della 0.4. Esempio:

In [4]:
def show_gold(rid):
    g = gold[rid]
    ids = {e["id"]: e["mention"] for e in g["entities"]}
    print(f"{rid}  {g['headline']}")
    print("   nodi  :", ", ".join(f"{e['mention']} [{'|'.join(e['types'])}]" for e in g["entities"] if not e["opt"]))
    print("   archi :", "; ".join(f"{'|'.join(ids[x] for x in c['subjects'])} —{c['predicate']}→ {'|'.join(ids[x] for x in c['objects'])}"
                                  for c in g["edges"] if not c["opt"]) or "— (solo nodi)")
for rid in ("r43", "r00", "r36"):
    show_gold(rid)
n_gold = {lvl: sum(c[(lvl, k)][0] + c[(lvl, k)][2] for c, _, _ in BD["gpt-4o"] for k in (S.NODE_KINDS_V04 if lvl == "nodo" else S.EDGE_KINDS_V04))
          for lvl in ("nodo", "arco")}
print(f"\nNel gold: {n_gold['nodo']} nodi e {n_gold['arco']} archi obbligatori (più alcuni facoltativi, che non contano né a favore né contro).")

r43  Iveco Signs EU755m Contract With Italy Army to Supply Trucks
   nodi  : Iveco [COMPANY], Italy Army [ORGANIZATION], Trucks [PRODUCT]
   archi : Iveco —SUPPLIES_TO→ Italy Army; Iveco —OFFERS→ Trucks
r00  *HAVELLS PARTNERS WITH JUMBO GROUP FOR KITCHEN APPLIANCES IN UAE
   nodi  : HAVELLS [COMPANY], JUMBO GROUP [COMPANY], KITCHEN APPLIANCES [PRODUCT], UAE [LOCATION]
   archi : HAVELLS —PARTNERS_WITH→ JUMBO GROUP; HAVELLS —ACTIVE_IN→ KITCHEN APPLIANCES; JUMBO GROUP —ACTIVE_IN→ KITCHEN APPLIANCES
r36  Intuitive Surgical Raised to Buy at Redburn; PT $510
   nodi  : Intuitive Surgical [COMPANY], Redburn [COMPANY]
   archi : — (solo nodi)

Nel gold: 93 nodi e 26 archi obbligatori (più alcuni facoltativi, che non contano né a favore né contro).


<a id="e1-risultati"></a>
### 5.1 Risultati: nodi e archi

- **Precision** = quanti dei nodi/archi prodotti sono giusti. **Recall** = quanti di quelli del gold sono stati trovati.
- Un nodo è giusto se nome e tipo coincidono; un arco se coincidono i due estremi e il tipo di freccia.
- Fra parentesi l'intervallo di confidenza al 95% dell'F1 (quanto potrebbe cambiare con altre 50 news).
- **Modalità giusta** = sugli archi trovati, quante volte fatto/piano/incerto/negato è corretto.

In [5]:
def totals(system, level, kinds):
    per = [tuple(sum(c[(level, k)][i] for k in kinds) for i in range(3)) for c, _, _ in BD[system]]
    lo, hi = S.bootstrap_f1([{"x": p} for p in per], "x")
    return S.prf(*map(sum, zip(*per))), lo, hi, per

rows = []
for s in SYSTEMS:
    row = {"sistema": s}
    for label, level, kinds in (("Nodi", "nodo", S.NODE_KINDS_V04), ("Archi", "arco", S.EDGE_KINDS_V04)):
        m, lo, hi, _ = totals(s, level, kinds)
        row[f"{label}: F1"] = f"{pct(m['f1'])} ({pct(lo)}–{pct(hi)})"
        row[f"{label}: precision"] = pct(m["precision"])
        row[f"{label}: recall"] = pct(m["recall"])
    ok, n = map(sum, zip(*[mod for _, _, mod in BD[s]]))
    row["modalità giusta"] = pct(ok / n) if s in X.MODELS and n else "–"
    rows.append(row)
display(pd.DataFrame(rows).set_index("sistema"))
print("GLiNER non assegna la modalità.\n")
for other in [m for m in X.MODELS if m != "gpt-4o"]:
    for label, level, kinds in (("nodi", "nodo", S.NODE_KINDS_V04), ("archi", "arco", S.EDGE_KINDS_V04)):
        a = [{"x": p} for p in totals(other, level, kinds)[3]]
        b = [{"x": p} for p in totals("gpt-4o", level, kinds)[3]]
        lo, hi = S.paired_diff(a, b, "x")
        d = totals(other, level, kinds)[0]["f1"] - totals("gpt-4o", level, kinds)[0]["f1"]
        print(f"{other} rispetto a gpt-4o, {label}: {d:+.0%} (IC {lo:+.0%} … {hi:+.0%}) → "
              f"{'non significativa' if lo <= 0 <= hi else 'significativa'}")

,Nodi: F1,Nodi: precision,Nodi: recall,Archi: F1,Archi: precision,Archi: recall,modalità giusta
sistema,,,,,,,
gpt-4o,94% (90%–97%),90%,98%,72% (54%–86%),59%,92%,100%
gpt-5.5,96% (93%–99%),94%,99%,91% (81%–98%),84%,100%,100%
qwen3-30b-a3b,78% (68%–86%),69%,89%,49% (33%–62%),32%,100%,100%
GLiNER inglese,63% (53%–71%),54%,75%,26% (9%–41%),22%,31%,–
GLiNER multilingue,54% (45%–63%),45%,69%,17% (7%–27%),12%,31%,–


GLiNER non assegna la modalità.

gpt-5.5 rispetto a gpt-4o, nodi: +3% (IC +0% … +5%) → significativa


gpt-5.5 rispetto a gpt-4o, archi: +20% (IC +5% … +36%) → significativa
qwen3-30b-a3b rispetto a gpt-4o, nodi: -16% (IC -25% … -8%) → significativa


qwen3-30b-a3b rispetto a gpt-4o, archi: -23% (IC -38% … -9%) → significativa


<a id="e1-tipo"></a>
### 5.2 F1 per tipo di nodo e di arco

`n` = quanti ce ne sono nel gold. Con meno di 5 casi il numero è un aneddoto, non una stima. Nessuna delle 50 news
contiene un `EXPOSED_TO` nel gold: quel tipo di freccia qui si può solo sbagliare (tutti falsi positivi).

In [6]:
rows = []
for level, kinds in (("nodo", S.NODE_KINDS_V04), ("arco", S.EDGE_KINDS_V04)):
    for k in kinds:
        row = {"tipo": f"{level} · {k}", "n": sum(c[(level, k)][0] + c[(level, k)][2] for c, _, _ in BD["gpt-4o"])}
        for s in SYSTEMS:
            tp, fp, fn = map(sum, zip(*[c[(level, k)] for c, _, _ in BD[s]]))
            row[s] = pct(S.prf(tp, fp, fn)["f1"]) if tp + fn else (f"– ({fp} falsi)" if fp else "–")
        rows.append(row)
display(pd.DataFrame(rows).set_index("tipo"))

,n,gpt-4o,gpt-5.5,qwen3-30b-a3b,GLiNER inglese,GLiNER multilingue
tipo,,,,,,
nodo · COMPANY,54,99%,100%,94%,80%,75%
nodo · ORGANIZATION,8,94%,94%,67%,64%,10%
nodo · PRODUCT,15,93%,97%,76%,56%,60%
nodo · CONCEPT,2,67%,67%,18%,15%,13%
nodo · SECTOR,4,89%,89%,86%,25%,44%
nodo · LOCATION,10,80%,91%,70%,69%,42%
arco · OFFERS,7,88%,100%,88%,40%,42%
arco · ACTIVE_IN,11,90%,85%,58%,27%,6%
arco · EXPOSED_TO,0,– (8 falsi),– (1 falsi),– (23 falsi),–,– (2 falsi)


<a id="e1-errori"></a>
### 5.3 Esempi di errori

Tutti gli errori sono stati riletti a mano e divisi in gradi: **grave** (freccia falsa o senza senso), **formale** (fatto
vero ma regola violata), **lieve** (nodo in più poco dannoso), **difendibile** (lettura plausibile che il gold non
accetta, vedi [da decidere](#decidere)). Esempi scelti per mostrare i tipi di errore:

In [7]:
EXAMPLES = [   # (sistema, id, errore come in ERR, grado, perché)
    ("gpt-4o", "r38", "Oaktree —ACQUIRES→ Banca Progetto", "grave", "Oaktree è il venditore: acquisizione rovesciata (nella 0.3 GPT-4o non lo faceva)"),
    ("gpt-4o", "r08", "Manufacturing —EXPOSED_TO→ Tankan", "grave", "un settore «esposto» al sondaggio che lo misura: EXPOSED_TO usato a caso"),
    ("gpt-4o", "r47", "Adidas —EXPOSED_TO→ Sambas", "formale", "vero nella sostanza, ma la news dice «amid», non «on» o «as»"),
    ("gpt-4o", "r38", "Italy [LOCATION]", "lieve (nodo)", "«Italy's Banca Progetto»: la nazionalità non è un nodo"),
    ("gpt-5.5", "r26", "R&F —ACTIVE_IN→ Court", "grave", "un tribunale preso per attività dell'azienda"),
    ("gpt-5.5", "r46", "Hanwha Systems —ACTIVE_IN→ Arms", "difendibile", "«Radar Order for Arms Exports»: è attiva negli armamenti; il gold non lo prevede"),
    ("qwen3-30b-a3b", "r01", "EUROAPI —EXPOSED_TO→ FY Net Sales", "grave", "«esposta» ai propri ricavi: 23 EXPOSED_TO falsi di questo tipo"),
    ("qwen3-30b-a3b", "r29", "SoftBank —OFFERS→ SoftBank", "grave", "un'azienda che «offre» sé stessa"),
    ("qwen3-30b-a3b", "r38", "Oaktree —ACQUIRES→ Banca Progetto", "grave", "acquisizione rovesciata, come GPT-4o"),
    ("GLiNER inglese", "r29", "SoftBank —ACQUIRES→ $3.4 Billion", "grave", "compra una cifra, non un'azienda"),
    ("GLiNER inglese", "r39", "Telkom Indonesia —ACQUIRES→ Huawei", "grave", "una partnership diventa l'acquisto di Huawei"),
    ("GLiNER inglese", "r18", "ELECTRIC GLASS [PRODUCT]", "grave (nodo)", "«Nippon Electric Glass» spezzato in due nodi"),
]
known = set(zip(ERR.sistema, ERR.id, ERR.dettaglio))
assert all((s, i, a) in known for s, i, a, *_ in EXAMPLES), [x for x in EXAMPLES if tuple(x[:3]) not in known]
heads = dict(zip(ERR.id, ERR.headline))
display(pd.DataFrame([{"modello": s, "news": heads[i], "cosa ha scritto": a, "grado": g, "perché": w}
                      for s, i, a, g, w in EXAMPLES]).set_index("modello"))
print("Errori per modello:", ", ".join(f"{s} {int((ERR.sistema == s).sum())}" for s in SYSTEMS))

,news,cosa ha scritto,grado,perché
modello,,,,
gpt-4o,Centerbridge Agrees to Buy Italy’s Banca Progetto from Oaktree,Oaktree —ACQUIRES→ Banca Progetto,grave,Oaktree è il venditore: acquisizione rovesciata (nella 0.3 GPT-4o non lo faceva)
gpt-4o,"BOJ Tankan Shows Manufacturing Weakens, Services Advances (1)",Manufacturing —EXPOSED_TO→ Tankan,grave,un settore «esposto» al sondaggio che lo misura: EXPOSED_TO usato a caso
gpt-4o,Adidas Lifts Forecast Amid Strong Sales of Sambas and Yeezys (1),Adidas —EXPOSED_TO→ Sambas,formale,"vero nella sostanza, ma la news dice «amid», non «on» o «as»"
gpt-4o,Centerbridge Agrees to Buy Italy’s Banca Progetto from Oaktree,Italy [LOCATION],lieve (nodo),«Italy's Banca Progetto»: la nazionalità non è un nodo
gpt-5.5,Chinese Developer R&F Faces Court Showdown Over Defaulted Loan,R&F —ACTIVE_IN→ Court,grave,un tribunale preso per attività dell'azienda
gpt-5.5,Hanwha Systems Wins $867m Radar Order for Arms Exports to Saudi,Hanwha Systems —ACTIVE_IN→ Arms,difendibile,«Radar Order for Arms Exports»: è attiva negli armamenti; il gold non lo prevede
qwen3-30b-a3b,*EUROAPI STILL SEES FY NET SALES -8% TO -11%,EUROAPI —EXPOSED_TO→ FY Net Sales,grave,«esposta» ai propri ricavi: 23 EXPOSED_TO falsi di questo tipo
qwen3-30b-a3b,SoftBank Unveils $3.4 Billion Buyback After Elliott’s Push (1),SoftBank —OFFERS→ SoftBank,grave,un'azienda che «offre» sé stessa
qwen3-30b-a3b,Centerbridge Agrees to Buy Italy’s Banca Progetto from Oaktree,Oaktree —ACQUIRES→ Banca Progetto,grave,"acquisizione rovesciata, come GPT-4o"


Errori per modello: gpt-4o 31, gpt-5.5 12, qwen3-30b-a3b 102, GLiNER inglese 129, GLiNER multilingue 187


<a id="e1-confronto"></a>
### 5.4 Confronto con la 0.3

Stesse 50 news, stessi modelli. Nella 0.3 contavano anche eventi e frecce ACTOR/TARGET; qui confronto le parti comuni:
entità/nodi e relazioni/archi. Attenzione: il gold 0.4 contiene regole nuove (per esempio `ACTIVE_IN` dallo scopo di un
accordo) decise **guardando gli output della 0.3 su queste stesse news**. Il confronto dice se la 0.4 è coerente e più
semplice da seguire, non quanto vale su news nuove.

In [8]:
gold03 = {g["id"]: g for g in X.load_jsonl("real_50_gold.jsonl")}
G03 = {"GLiNER inglese": load_gliner("gliner_044.jsonl"), "GLiNER multilingue": load_gliner("gliner_044_multi.jsonl")}
rows = []
for s in SYSTEMS:
    llm = s in X.MODELS
    b03 = [S.breakdown(X.extract(r["headline"], r["date"], s, True) if llm else G03[s][("real", r["id"])], gold03[r["id"]],
                       has_events=llm) for r in real]
    f = lambda bd, level, kinds: pct(S.prf(*map(sum, zip(*[tuple(sum(c[(level, k)][i] for k in kinds) for i in range(3)) for c, *_ in bd])))["f1"])
    rows.append({"sistema": s,
                 "0.3 · entità F1": f(b03, "nodo", S.NODE_KINDS[:-1]), "0.3 · relazioni F1": f(b03, "arco", S.EDGE_KINDS[:-1]),
                 "0.3 · errori": sum(len(e) for _, e in b03),
                 "0.4 · nodi F1": f(BD[s], "nodo", S.NODE_KINDS_V04), "0.4 · archi F1": f(BD[s], "arco", S.EDGE_KINDS_V04),
                 "0.4 · errori": int((ERR.sistema == s).sum())})
display(pd.DataFrame(rows).set_index("sistema"))
print("Errori = tutti i nodi e archi sbagliati o mancati; nella 0.3 includono eventi e frecce ACTOR/TARGET.")

,0.3 · entità F1,0.3 · relazioni F1,0.3 · errori,0.4 · nodi F1,0.4 · archi F1,0.4 · errori
sistema,,,,,,
gpt-4o,87%,83%,74,94%,72%,31
gpt-5.5,88%,79%,61,96%,91%,12
qwen3-30b-a3b,71%,27%,227,78%,49%,102
GLiNER inglese,66%,17%,131,63%,26%,129
GLiNER multilingue,58%,15%,171,54%,17%,187


Errori = tutti i nodi e archi sbagliati o mancati; nella 0.3 includono eventi e frecce ACTOR/TARGET.


<a id="e2"></a>
## 6. E2 — Leakage su 50 news trappola

Ogni trappola è una news inventata su aziende reali, datata prima di un fatto che oggi conosciamo (un'acquisizione, un
cambio di nome, un prodotto, un fallimento). Tutti i fatti sono pubblici prima di settembre 2023, quindi entrambi i GPT
possono conoscerli. Facciamo al modello tre richieste diverse; ecco le tre sulla stessa trappola:

In [9]:
P = {p["id"]: p for p in probes}
p = P["p41"]
h, d = p["headline"], p["as_of"]
print(f"NEWS del {d}: {h}\nFATTO FUTURO: {p['future_fact']} (pubblico dal {p['first_public']})\n")
print("1 · DOMANDA DIRETTA — «cosa è successo dopo?» (serve solo a sapere se il modello conosce il futuro)")
print("    gpt-4o:", X.knowledge(h, d, "gpt-4o"), "\n")
print("2 · ESTRAZIONE CON NOME UFFICIALE E TICKER — il prompt «ingenuo» di 0.42, che non vieta di usare la memoria")
for dated in (False, True):
    o = X.extract_naive(h, d, "gpt-4o", dated)
    print(f"    gpt-4o {'con data  ' if dated else 'senza data'}:", "; ".join(f"{e['text']} → {e['canonical_name']} ({e['ticker']})" for e in o["entities"]))
print("\n3 · IL NOSTRO ESTRATTORE (schema 0.4) — copia i nomi dalla news, la guideline vieta di completare da memoria")
for dated in (False, True):
    o = X.extract_v04(h, d, "gpt-4o", dated)
    print(f"    gpt-4o {'con data  ' if dated else 'senza data'}:", ", ".join(f"{n['mention']} [{n['type']}]" for n in o["nodes"]))

NEWS del 2019-06-18: Facebook Unveils Libra Cryptocurrency With Partners Including Visa
FATTO FUTURO: Libra renamed Diem; Facebook renamed Meta (pubblico dal 2020-12-01)

1 · DOMANDA DIRETTA — «cosa è successo dopo?» (serve solo a sapere se il modello conosce il futuro)
    gpt-4o: After the announcement of Libra in 2019, the project faced significant regulatory scrutiny and backlash from governments and financial institutions worldwide. Many initial partners, including Visa, eventually withdrew from the project. In December 2020, the Libra Association rebranded to Diem, but the project struggled to gain traction and was ultimately shut down in early 2022. 

2 · ESTRAZIONE CON NOME UFFICIALE E TICKER — il prompt «ingenuo» di 0.42, che non vieta di usare la memoria
    gpt-4o senza data: Facebook → Meta Platforms, Inc. (META); Libra → Libra (None); Visa → Visa Inc. (V)
    gpt-4o con data  : Facebook → Facebook, Inc. (FB); Visa → Visa Inc. (V)

3 · IL NOSTRO ESTRATTORE (schema 0.4) — co

<a id="e2-risultati"></a>
### 6.1 Risultati

Numero di trappole (su 50) in cui il modello scrive il fatto futuro. Nella prima riga è la risposta attesa (il modello
lo conosce); nelle altre due è un leak.

In [10]:
rows = []
for label, kind in (("«Cosa è successo dopo?» — conosce il fatto", "diretta"),
                    ("Estrazione con nome ufficiale e ticker — leak", "ticker"),
                    ("Nostro estrattore (0.4) — leak", "estrattore")):
    row = {"richiesta": label}
    for s in SYSTEMS:
        if (kind, s, None) in E2:
            row[s] = str(sum(E2[(kind, s, None)]))
        elif (kind, s, False) in E2:
            row[s] = f"{sum(E2[(kind, s, False)])} senza data / {sum(E2[(kind, s, True)])} con data"
        else:
            row[s] = "–"
    rows.append(row)
display(pd.DataFrame(rows).set_index("richiesta"))
fake = {"entities": [{"id": "e1", "mention": "Figma"}], "events": [{"id": "v1", "trigger": "Valued", "modality": "reported",
        "description": "Figma, later bought by Adobe, raised money"}], "claims": []}
print("Controllo del rilevatore: un leak finto («Figma, later bought by Adobe») viene trovato →", S.leak_flags(fake, P["p21"])["any"])

,gpt-4o,gpt-5.5,qwen3-30b-a3b,GLiNER inglese,GLiNER multilingue
richiesta,,,,,
«Cosa è successo dopo?» — conosce il fatto,38,42,34,–,–
Estrazione con nome ufficiale e ticker — leak,6 senza data / 1 con data,8 senza data / 1 con data,6 senza data / 3 con data,–,–
Nostro estrattore (0.4) — leak,0 senza data / 0 con data,0 senza data / 0 con data,0 senza data / 0 con data,0,0


Controllo del rilevatore: un leak finto («Figma, later bought by Adobe») viene trovato → True


<a id="e2-esempi"></a>
### 6.2 Esempi di leak

Con la richiesta "nome ufficiale e ticker" i GPT fanno trapelare il futuro solo nei **nomi**; Qwen anche nelle
**relazioni**, cioè scrive come fatto un'acquisizione avvenuta dopo la data della news.

In [11]:
def leak_example(m, pid, how):
    pr = P[pid]
    if how == "estrattore":
        o = X.extract_v04(pr["headline"], pr["as_of"], m, True)
        assert S.leak_flags(S.as_v03(o), pr)["any"]
        nm = {n["id"]: n["mention"] for n in o["nodes"]}
        wrote = "; ".join(f"{nm.get(e['subject_id'])} —{e['predicate']}→ {nm.get(e['object_id'])} [{e['modality']}]" for e in o["edges"])
        req = "nostro estrattore, con data"
    else:
        o = X.extract_naive(pr["headline"], pr["as_of"], m, how == "con data")
        f = S.leak_flags_naive(o, pr)
        assert f["any"]
        r = o["relation"]
        wrote = (f"{r['subject']} {r['predicate']} {r['object']} [{r['deal_status']}]" if f["pair"] else
                 "; ".join(f"{e['text']} → {e['canonical_name']}" for e in o["entities"] if e["canonical_name"] != e["text"]))
        req = f"nome ufficiale e ticker, {how}"
    return {"modello": m, "richiesta": req, "news (data)": f"{pr['headline']} ({pr['as_of']})", "cosa ha scritto": wrote,
            "fatto futuro (pubblico dal)": f"{pr['future_fact']} ({pr['first_public']})"}

LEAK_EXAMPLES = [("gpt-4o", "p41", "senza data"), ("gpt-5.5", "p42", "senza data"),
                 ("qwen3-30b-a3b", "p13", "senza data"), ("qwen3-30b-a3b", "p13", "con data"), ("qwen3-30b-a3b", "p19", "con data")]
LEAK_EXAMPLES += [(m, p["id"], "estrattore") for m in X.MODELS for p, h in zip(probes, E2[("estrattore", m, True)]) if h]
display(pd.DataFrame([leak_example(*x) for x in LEAK_EXAMPLES]).set_index("modello"))

,richiesta,news (data),cosa ha scritto,fatto futuro (pubblico dal)
modello,,,,
gpt-4o,"nome ufficiale e ticker, senza data",Facebook Unveils Libra Cryptocurrency With Partners Including Visa (2019-06-18),"Facebook → Meta Platforms, Inc.; Visa → Visa Inc.",Libra renamed Diem; Facebook renamed Meta (2020-12-01)
gpt-5.5,"nome ufficiale e ticker, senza data",Square Shares Jump as Cash App Revenue Surges (2020-02-26),"Square → Block, Inc.",Square renamed Block (2021-12-01)
qwen3-30b-a3b,"nome ufficiale e ticker, senza data","Microsoft, Activision Blizzard Lead Gaming Stocks Lower (2021-11-15)",Microsoft acquires Activision Blizzard [completed],Microsoft to buy Activision (2022-01-18)
qwen3-30b-a3b,"nome ufficiale e ticker, con data","Microsoft, Activision Blizzard Lead Gaming Stocks Lower (2021-11-15)",Microsoft acquires Activision Blizzard [announced],Microsoft to buy Activision (2022-01-18)
qwen3-30b-a3b,"nome ufficiale e ticker, con data","AMD, Xilinx Gain as Chip Stocks Extend Rally (2020-06-01)",AMD acquires Xilinx [announced],AMD to buy Xilinx (2020-10-08)


<a id="conclusioni"></a>
## 7. Conclusioni

1. **Lo schema 0.4 è più semplice e i modelli lo seguono meglio.** Meno tipi e meno regole danno molti meno errori
   (GPT-5.5: 12 invece di 61), senza perdere i collegamenti che servono per i temi (recall degli archi 92–100%).
2. **Estrattore: GPT-5.5.** Con la 0.4 il distacco da GPT-4o diventa netto sugli archi (+20 punti, significativo). Costa
   circa 2,4 volte GPT-4o (tabella di 0.43 §9), ma il prompt 0.4 è più corto: ~1.940 token di input invece di ~2.700.
3. **Cosa sistemare nel prompt**: (a) un esempio contro `EXPOSED_TO` usato a sproposito ("amid", sondaggi, variabili
   finanziarie), l'errore principale di GPT-4o e Qwen; (b) un esempio sulle nazionalità ("Italy's", "European", "India's"),
   ancora estratte come luoghi; (c) "X to buy Y from Z": Z è il venditore, non compra.
4. **Leakage**: il nostro formato lo blocca per tutti i modelli; la data da sola lo riduce ma non lo azzera; Qwen, se gli
   si chiede nome ufficiale e ticker, scrive anche acquisizioni avvenute dopo la data.
5. **Qwen e GLiNER** così come sono non sostituiscono GPT. **Suggerimento: fine-tuning di Qwen3-30B-A3B sulle etichette di
   GPT-5.5**, con la guideline 0.4, su qualche migliaio di news (per esempio 5.000–20.000), poi rimisurato su questo gold e
   sulle trappole. Qwen trova già tutte le relazioni vere (recall 100%): l'addestramento deve insegnargli soprattutto a
   *non* aggiungere frecce (`EXPOSED_TO` e `ACTIVE_IN` a caso). Il vantaggio è la scala: gira in locale, senza costo per
   chiamata, sui 22,8 milioni di news. Lo stesso si può provare con GLiNER, più piccolo e veloce ma molto più lontano oggi.
   Le etichette di GPT sono copiate dal testo, quindi non trasmettono al modello addestrato fatti dal futuro.
6. **Prossimo passo**: un test su 50–100 news nuove, ricche di prodotti, tecnologie e movimenti con una causa ("on", "as"),
   annotate prima di vedere gli output. È l'unico modo di misurare davvero `EXPOSED_TO` e le regole nuove della 0.4.

<a id="distillation"></a>
### 7.1 Distillation: quanto tempo serve

**Distillation** = GPT-5.5 (l'"insegnante") etichetta alcune migliaia di news con lo schema 0.4; Qwen3-30B-A3B (lo
"studente") viene addestrato su quelle etichette con LoRA (si addestra un piccolo strato aggiuntivo, non tutto il modello);
poi lo si rimisura su E1 ed E2. Dopo l'addestramento la guideline non serve più nel prompt.

**Tempi stimati, non misurati** (nessuna prova di addestramento è stata ancora fatta), con 10.000 news:

| Passo | Mac (M5 Pro, 48 GB) | A100 |
|---|---|---|
| 1. Etichette di GPT-5.5 | ~35 min (API) | ~35 min (API) |
| 2. Addestramento LoRA, 3 epoche | **4–14 ore (una notte)** | **1–3 ore** |
| 3. Preparazione del modello addestrato (unire LoRA, convertire) | ~30 min | ~15 min |
| 4. Rimisura su 50 news + 50 trappole | ~5 min | ~5 min |
| **Totale** | **circa una notte** | **mezza giornata** |

Il tempo di addestramento cresce in proporzione al numero di news:

| News etichettate | Mac | A100 |
|---|---|---|
| 5.000 | 2–7 ore | 30 min – 1,5 ore |
| 10.000 | 4–14 ore | 1–3 ore |
| 20.000 | 8–28 ore | 2–6 ore |

Ipotesi: circa 250 token per esempio (news + risposta, senza guideline), 3 epoche, cioè circa 7,5 milioni di token per
10.000 news; 1.000–3.000 token/s su A100 e 150–500 token/s sul Mac con MLX e il modello a 4 bit. Qwen3-30B-A3B è un modello
MoE, che gli strumenti di addestramento gestiscono meno bene dei modelli densi: da qui gli intervalli larghi. Sul Mac MLX va
installato e la memoria è al limite. Un modello denso più piccolo (Qwen3-4B o 8B) si addestra 3–5 volte più in fretta e
potrebbe bastare, visto che il compito è quasi tutto estrattivo. Una prova di mezz'ora sul Mac (circa 300 news etichettate,
qualche centinaio di passi) sostituirebbe queste stime con numeri misurati.

<a id="decidere"></a>
## 8. Da decidere

Rispetto alla 0.3 le decisioni aperte sono poche; per ciascuna c'è una proposta.

| Caso | Esempio | Proposta |
|---|---|---|
| `ACTIVE_IN` verso un luogo quando la news dice dove opera l'azienda | "Remy … Eyes US Recovery" → Remy ─▶ US; "Raymond … Launches New Mumbai Project" → Raymond ─▶ Mumbai | accettarlo: è il vecchio *Operate In* di FinDKG, e i luoghi restano solo contesto |
| Sviluppare un impianto: `ACTIVE_IN` o `INVESTS_IN`? | "VGP Partners With Verne to Develop Autonomous Vehicle Plant": GPT-4o scrive `INVESTS_IN` | `ACTIVE_IN`, perché la news non dichiara una spesa |
| Lo scopo dichiarato vale anche per un ordine | "Hanwha Systems Wins Radar Order for Arms Exports" → Hanwha ─▶ Arms | accettarlo: è lo stesso criterio dello scopo di un accordo |
| `EXPOSED_TO` con "as" | nessun caso nelle 50 news | lasciare la regola ("on", "as" sì; "after", "amid" no) e misurarla sul test nuovo |

<a id="metodo"></a>
## 9. Come è stato fatto

- **Campione E1** ([sample_044.py](../scripts/sample_044.py)): 25 news casuali del 2024 + 25 scelte per parole-chiave legate ai
  predicati (seed 44), escluse le 150 di 0.42. Nessun modello ha scelto i casi; nessuna news sostituita.
- **Gold 0.4** ([real_50_gold_v04.py](gold/0.44/real_50_gold_v04.py)): ricavato dal gold 0.3 (un solo annotatore, Claude, scritto
  prima di vedere gli output) togliendo eventi e indicatori, più le regole decise per la 0.4 (`ACTIVE_IN` dallo scopo di un
  accordo, `OFFERS` per "vince un ordine di…", entrambe le letture quando due sono valide). Queste regole sono state decise
  **dopo** aver letto gli output della 0.3 su queste news: E1 sulla 0.4 è una misura di sviluppo, non un test pulito.
- **Guideline** [044-v2](gold/0.44/guideline_v04.md), la stessa per gold e modelli; quattro esempi inventati nel prompt.
- **Scoring** ([eval_044.py](../scripts/eval_044.py), `breakdown_v04`): nomi confrontati dopo normalizzazione (uguali, o uno
  contenuto nell'altro); abbinamento uno-a-uno; un tipo sbagliato conta come un errore in più e uno in meno; gli elementi
  facoltativi del gold non contano.
- **Modelli**: GPT-4o = `gpt-4o-2024-08-06`, temperatura 0; GPT-5.5 = `gpt-5.5-2026-04-23`, reasoning `none`; schema JSON imposto.
  **Qwen3-30B-A3B-Instruct-2507** in locale su questo Mac (Apple M5 Pro, 48 GB): `unsloth/…-Q4_K_M.gguf` (4 bit, 18,6 GB,
  sha256 `6c997b8a…`), llama.cpp build 10330, temperatura 0, schema imposto dalla grammatica. Per riaccenderlo, da `code/`:
  `llama-server -m data/models/Qwen3-30B-A3B-Instruct-2507-Q4_K_M.gguf -c 32768 -np 4 --jinja`.
  **GLiNER2.5** ([gliner_044.py](../scripts/gliner_044.py) `base v04` / `multi v04`): JointIE con i 6 tipi di nodo e i 7 archi,
  soglie di default, in un ambiente separato (`.venv-gliner`) perché richiede transformers 4.x.
- **Trappole E2** ([synthetic_50.py](gold/0.44/synthetic_50.py)): news inventate, datate prima di un fatto pubblico entro il
  2023-09-01. Le date dei fatti vengono dalla conoscenza dell'annotatore, controllate per ordine cronologico, non su fonte
  primaria. Un leak è una parola-firma del fatto futuro presente nell'output ma non nella news, una relazione di deal fra
  le due aziende citate, o lo stato finale di un deal. Le richieste "cosa è successo dopo?" e "nome ufficiale e ticker"
  sono indipendenti dallo schema e sono le stesse della 0.3.
- **Modifiche fatte dopo aver visto gli output** (tutte uguali per ogni sistema): le regole della 0.4 sopra; nella 0.3,
  l'allineamento degli eventi senza verbo e le due richieste di controllo in E2; il nome corretto alla data per p46
  ("Royal Dutch Shell plc"). I risultati della 0.3 restano in cache e nel [confronto](#e1-confronto).

<a id="dati"></a>
## 10. Dati completi

Le tabelle complete sono lunghe e non vengono mostrate. Metti `MOSTRA = True` ed esegui la cella per vederle:
le 50 news con il gold, tutti gli errori di ogni modello, le 50 trappole, tutti i leak e le risposte alla domanda diretta.

In [12]:
MOSTRA = False

if not MOSTRA:
    print("Tabelle complete nascoste. Variabili disponibili: ERR (tutti gli errori E1), E2 (trappole segnalate), gold, probes.")
else:
    with pd.option_context("display.max_rows", 2000):
        print("50 NEWS REALI E GOLD 0.4")
        for r in real:
            show_gold(r["id"])
        for s in SYSTEMS:
            e = ERR[ERR.sistema == s]
            print(f"\nERRORI · {s}: {len(e)} ({(e.esito == 'FP').sum()} prodotti ma non nel gold, {(e.esito == 'FN').sum()} mancati)")
            display(e.set_index(["id", "headline"])[["livello", "tipo", "esito", "categoria", "dettaglio"]])
        print("\n50 TRAPPOLE")
        display(pd.DataFrame([dict(id=p["id"], famiglia=p["family"], data=p["as_of"], news=p["headline"],
                                   fatto_futuro=p["future_fact"], pubblico_dal=p["first_public"]) for p in probes]).set_index("id"))
        print("\nTRAPPOLE CON UN LEAK")
        display(pd.DataFrame([dict(richiesta=k[0], sistema=k[1], data="—" if k[2] is None else ("con" if k[2] else "senza"),
                                   id=p["id"], news=p["headline"])
                              for k, hits in E2.items() if k[0] != "diretta" for p, h in zip(probes, hits) if h]))
        print("\nDOMANDA DIRETTA (gpt-5.5)")
        display(pd.DataFrame([dict(id=p["id"], news=p["headline"], risposta=X.knowledge(p["headline"], p["as_of"], "gpt-5.5"))
                              for p in probes]).set_index("id"))

Tabelle complete nascoste. Variabili disponibili: ERR (tutti gli errori E1), E2 (trappole segnalate), gold, probes.
